### Forecasting HPC Workload Intensity with LSTM-EDAIN

This notebook forecasts `num_jobs` from the LLNL BlueGene/L Parquet workload trace using an LSTM with Extended Deep Adaptive Input Normalization (EDAIN) on inputs, trained with Adam.

**Inputs:** train-fit global `StandardScaler`, then EDAIN (global-aware).  
**Target:** same Approach A as DAIN/vanilla — raw per-window z-score of historical `num_jobs`.

Set `FEATURE_MODE` to `'univariate'` or `'multivariate'` in the feature-selection cell to switch input features.


### EDAIN implementation note
Keras translation of the global-aware EDAIN layer from [marcusGH/edain_paper](https://github.com/marcusGH/edain_paper), with separate learning-rate modifiers for outlier, shift, scale, and power sublayers.

The global scaler is applied to **inputs only** (paper-faithful global-aware EDAIN). The forecast target is normalized/inverted with raw per-window statistics of the historical target channel (Approach A), recovered from the scaled input via the fitted scaler parameters.


## Step 1: Library Imports

In [ ]:
# Import necessary libraries
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')
%matplotlib inline

SEED = 42
tf.keras.utils.set_random_seed(SEED)

SLICE_MINUTES = 15
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.85
WINDOW_CANDIDATES = [30, 60, 90, 120]
BATCH_SIZE = 32
EPOCHS = 50
BASE_LR = 1e-3
EDAIN_LR_SCALES = {'outlier': 100.0, 'shift': 0.01, 'scale': 0.01, 'power': 10.0}
RAW_STD_FLOOR = 1e-3
DATASET_PATH = "/kaggle/working/25-LLNL-uBGL-2006-2.parquet"


## Step 2: Data Preparation and Splitting

In [ ]:
import pandas as pd
import numpy as np

def preprocess_data(path, slice_minutes):
    jobs = pd.read_parquet(path, columns=['submit_time', 'run_time', 'allocated_proc'])
    jobs = jobs.astype({'run_time': 'float64', 'allocated_proc': 'float64'})
    unix_time = (jobs['submit_time'] - pd.Timestamp('1970-01-01', tz='UTC')).dt.total_seconds()
    jobs['submit_time'] = unix_time - unix_time.min()
    jobs = jobs[jobs['run_time'] > 0].copy()
    jobs['slice'] = (jobs['submit_time'] // (slice_minutes * 60)).astype('int64')
    data_agg = jobs.groupby('slice').agg(
        num_jobs=('run_time', 'size'),
        total_run_time=('run_time', 'sum'),
        std_run_time=('run_time', 'std'),
        total_allocated_proc=('allocated_proc', 'sum'),
        std_allocated_proc=('allocated_proc', 'std'),
    )
    data_agg['avg_allocated_proc'] = data_agg['total_allocated_proc'] / data_agg['num_jobs']
    data_agg['avg_run_time'] = data_agg['total_run_time'] / data_agg['num_jobs']
    full_range = np.arange(data_agg.index.min(), data_agg.index.max() + 1)
    data_agg = data_agg.reindex(full_range, fill_value=0).fillna(0)
    return data_agg.reset_index(drop=True)[
        ['num_jobs', 'total_allocated_proc', 'std_allocated_proc', 'total_run_time',
         'std_run_time', 'avg_allocated_proc', 'avg_run_time']
    ]

df = preprocess_data(DATASET_PATH, SLICE_MINUTES)
df

In [ ]:
# Feature mode switch: 'univariate' uses only the target; 'multivariate' uses all listed covariates.
FEATURE_MODE = 'multivariate'  # 'univariate' | 'multivariate'

target_col = 'num_jobs'
MULTIVARIATE_FEATURE_COLS = [
    'num_jobs',
    'total_allocated_proc',
    'std_allocated_proc',
    'total_run_time',
    'std_run_time',
    'avg_allocated_proc',
    'avg_run_time',
]

if FEATURE_MODE == 'univariate':
    feature_cols = [target_col]
elif FEATURE_MODE == 'multivariate':
    feature_cols = list(MULTIVARIATE_FEATURE_COLS)
else:
    raise ValueError("FEATURE_MODE must be 'univariate' or 'multivariate'")

if target_col not in feature_cols:
    raise ValueError(f"target_col '{target_col}' must be included in feature_cols")

data = df[feature_cols].dropna()

target_idx = feature_cols.index(target_col)
n_features = len(feature_cols)

print("Feature mode:", FEATURE_MODE)
print("Features:", feature_cols)
print("Target:", target_col)
print("Number of features:", n_features)
print("Target index:", target_idx)

### Data scaling / normalization
A `StandardScaler` is fit on **training rows only** and applied to the full feature matrix for **inputs** (global-aware EDAIN).

Target loss/metrics use Approach A: per-window raw z-score of historical `num_jobs`, not the globally scaled target values.


In [ ]:
# Raw series + chronological split
dataset_raw = data.values.astype(np.float32)
train_end = int(np.ceil(len(dataset_raw) * TRAIN_FRACTION))
val_end = int(np.ceil(len(dataset_raw) * VALIDATION_FRACTION))

scaler = StandardScaler().fit(dataset_raw[:train_end])
scaled_data = scaler.transform(dataset_raw).astype(np.float32)

# Scaler params for recovering original-scale target channel from scaled X
TARGET_MEAN = np.float32(scaler.mean_[target_idx])
TARGET_SCALE = np.float32(scaler.scale_[target_idx])

print(f"Training: {train_end} | Validation: {val_end - train_end} | Test: {len(dataset_raw) - val_end}")
print(f"Target scaler mean={TARGET_MEAN:.4f}, scale={TARGET_SCALE:.4f}")
print('Inputs: global StandardScaler + EDAIN | Target: Approach A raw per-window z-score')


## Step 3: EDAIN Layer

In [ ]:
class AdaptiveOutlierRemoval(tf.keras.layers.Layer):
    def __init__(self, residual_connection=True, mode='exp', min_beta=1.0, init_sigma=0.1, **kwargs):
        super().__init__(**kwargs)
        if mode not in ('exp', 'softplus'):
            raise ValueError(f"Unknown outlier removal mode: {mode}")
        self.residual_connection = residual_connection
        self.mode = mode
        self.min_beta = min_beta
        self.init_sigma = init_sigma

    def build(self, input_shape):
        window_size, dim = input_shape[1], input_shape[-1]
        self.log_cutoff = self.add_weight(
            name='log_cutoff', shape=(dim,),
            initializer=tf.keras.initializers.RandomNormal(1.0, self.init_sigma))
        if self.residual_connection:
            self.alpha = self.add_weight(
                name='alpha', shape=(dim,),
                initializer=tf.keras.initializers.RandomNormal(0.0, self.init_sigma))
        self.running_mean = self.add_weight(
            name='running_mean', shape=(window_size, dim), initializer='zeros', trainable=False)
        self.running_n = self.add_weight(
            name='running_n', shape=(), initializer='zeros', trainable=False)

    def call(self, x, training=False):
        mean = tf.convert_to_tensor(self.running_mean)
        if training:
            batch = tf.cast(tf.shape(x)[0], x.dtype)
            total = self.running_n + batch
            mean = (mean * self.running_n + tf.reduce_sum(tf.stop_gradient(x), axis=0)) / total
            self.running_mean.assign(mean)
            self.running_n.assign(total)

        if self.mode == 'exp':
            beta = self.min_beta + tf.exp(self.log_cutoff)
        else:
            beta = self.min_beta + tf.nn.softplus(self.log_cutoff)

        x_tanh = beta * tf.tanh((x - mean) / beta) + mean
        if self.residual_connection:
            gate = tf.sigmoid(self.alpha)
            return (1.0 - gate) * x + gate * x_tanh
        return x_tanh


class AdaptiveShift(tf.keras.layers.Layer):
    def __init__(self, init_sigma=0.1, **kwargs):
        super().__init__(**kwargs)
        self.init_sigma = init_sigma

    def build(self, input_shape):
        self.shift = self.add_weight(
            name='shift', shape=(input_shape[-1],),
            initializer=tf.keras.initializers.RandomNormal(0.0, self.init_sigma))

    def call(self, x, training=False):
        return x - self.shift


class AdaptiveScale(tf.keras.layers.Layer):
    def __init__(self, init_sigma=0.1, **kwargs):
        super().__init__(**kwargs)
        self.init_sigma = init_sigma

    def build(self, input_shape):
        self.log_scale = self.add_weight(
            name='log_scale', shape=(input_shape[-1],),
            initializer=tf.keras.initializers.RandomNormal(0.0, self.init_sigma))

    def call(self, x, training=False):
        return x * tf.exp(-self.log_scale)


class AdaptivePowerTransform(tf.keras.layers.Layer):
    def __init__(self, init_sigma=0.1, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.init_sigma = init_sigma
        self.eps = eps

    def build(self, input_shape):
        self.lambd = self.add_weight(
            name='lambd', shape=(input_shape[-1],),
            initializer=tf.keras.initializers.RandomNormal(1.0, self.init_sigma))

    def call(self, x, training=False):
        lam = tf.convert_to_tensor(self.lambd)
        abs_x = tf.abs(x)
        is_l0 = tf.abs(lam) < self.eps
        is_l2 = tf.abs(lam - 2.0) < self.eps
        safe_l0 = tf.where(is_l0, tf.ones_like(lam), lam)
        safe_l2 = tf.where(is_l2, tf.ones_like(lam), lam - 2.0)

        positive = tf.where(
            is_l0, tf.math.log1p(abs_x), (tf.pow(abs_x + 1.0, safe_l0) - 1.0) / safe_l0)
        negative = tf.where(
            is_l2, -tf.math.log1p(abs_x), (tf.pow(abs_x + 1.0, -safe_l2) - 1.0) / safe_l2)

        return tf.where(x >= 0.0, positive, negative)


class EDAINLayer(tf.keras.layers.Layer):
    def __init__(self, residual_connection=True, outlier_mode='exp', init_sigma=0.1, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.outlier_removal = AdaptiveOutlierRemoval(residual_connection, outlier_mode, init_sigma=init_sigma)
        self.shift = AdaptiveShift(init_sigma)
        self.scale = AdaptiveScale(init_sigma)
        self.power_transform = AdaptivePowerTransform(init_sigma, eps)

    def call(self, x, training=False):
        x = self.outlier_removal(x, training=training)
        x = self.shift(x, training=training)
        x = self.scale(x, training=training)
        return self.power_transform(x, training=training)

    def sublayers(self):
        return {
            'outlier': self.outlier_removal,
            'shift': self.shift,
            'scale': self.scale,
            'power': self.power_transform,
        }

## Step 4: Forecasting Model
EDAIN feeds an LSTM backbone. Target uses Approach A (raw per-window z-score). Because inputs are globally scaled, the historical target channel is unscaled with the fitted scaler before computing window µ/σ.


In [ ]:
def clone_optimizer_with_lr(optimizer, learning_rate):
    config = optimizer.get_config()
    config['learning_rate'] = learning_rate
    return optimizer.__class__.from_config(config)


def recover_raw_target_window(x_scaled, target_idx, target_mean, target_scale):
    """Undo global scaling on the target channel only (Approach A stats need raw units)."""
    return x_scaled[:, :, target_idx:target_idx + 1] * target_scale + target_mean


def raw_target_stats_from_scaled(x_scaled, target_idx, target_mean, target_scale, eps=RAW_STD_FLOOR):
    target_window = recover_raw_target_window(x_scaled, target_idx, target_mean, target_scale)
    raw_avg = tf.reduce_mean(target_window, axis=1)
    raw_std = tf.sqrt(
        tf.reduce_mean(tf.square(target_window - tf.expand_dims(raw_avg, 1)), axis=1) + 1e-8
    )
    raw_std = tf.maximum(raw_std, eps)
    return raw_avg, raw_std


def raw_target_stats_from_scaled_numpy(x_scaled, target_idx, target_mean, target_scale, eps=RAW_STD_FLOOR):
    target_window = x_scaled[:, :, target_idx:target_idx + 1] * target_scale + target_mean
    raw_avg = np.mean(target_window, axis=1)
    raw_std = np.sqrt(np.mean(np.square(target_window - np.expand_dims(raw_avg, 1)), axis=1) + 1e-8)
    raw_std = np.maximum(raw_std, eps)
    return raw_avg, raw_std


def normalize_target_raw(y, raw_avg, raw_std):
    return (y - raw_avg) / raw_std


def inverse_target_raw_numpy(normalized, raw_avg, raw_std):
    return normalized * raw_std + raw_avg


class EDAINForecastModel(tf.keras.Model):
    """EDAIN (inputs) + LSTM with Approach A raw per-window target z-score."""
    def __init__(self, edain_layer, backbone_layers, target_idx, target_mean, target_scale, **kwargs):
        super().__init__(**kwargs)
        self.edain = edain_layer
        self.backbone_layers = backbone_layers
        self.target_idx = target_idx
        self.target_mean = tf.constant(target_mean, dtype=tf.float32)
        self.target_scale = tf.constant(target_scale, dtype=tf.float32)
        self.loss_tracker = tf.keras.metrics.Mean(name='loss')

    def call(self, inputs, training=False):
        x = self.edain(inputs, training=training)
        for layer in self.backbone_layers:
            x = layer(x, training=training)
        return x

    def compile(self, optimizer, base_lr, lr_scales, **kwargs):
        super().compile(optimizer=optimizer, **kwargs)
        self.loss_fn = tf.keras.losses.MeanSquaredError()
        self.edain_optimizers = {
            name: clone_optimizer_with_lr(optimizer, base_lr * scale)
            for name, scale in lr_scales.items()
        }

    @property
    def metrics(self):
        return [self.loss_tracker]

    @staticmethod
    def apply_group(optimizer, grads, variables):
        pairs = [(g, v) for g, v in zip(grads, variables) if g is not None]
        if pairs:
            optimizer.apply_gradients(pairs)

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            pred = self(x, training=True)
            raw_avg, raw_std = raw_target_stats_from_scaled(
                x, self.target_idx, self.target_mean, self.target_scale
            )
            y_norm = normalize_target_raw(y, raw_avg, raw_std)
            loss = self.loss_fn(y_norm, pred)
            if self.losses:
                loss += tf.add_n(self.losses)

        variables = self.trainable_variables
        grads = dict(zip((id(v) for v in variables), tape.gradient(loss, variables)))

        edain_ids = set()
        for name, sublayer in self.edain.sublayers().items():
            group = list(sublayer.trainable_variables)
            edain_ids.update(id(v) for v in group)
            self.apply_group(self.edain_optimizers[name], [grads[id(v)] for v in group], group)

        main = [v for v in variables if id(v) not in edain_ids]
        self.apply_group(self.optimizer, [grads[id(v)] for v in main], main)

        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}

    def test_step(self, data):
        x, y = data
        pred = self(x, training=False)
        raw_avg, raw_std = raw_target_stats_from_scaled(
            x, self.target_idx, self.target_mean, self.target_scale
        )
        y_norm = normalize_target_raw(y, raw_avg, raw_std)
        loss = self.loss_fn(y_norm, pred)
        if self.losses:
            loss += tf.add_n(self.losses)
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}


training_callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
]


def build_model(window_size):
    backbone_layers = [
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ]
    model = EDAINForecastModel(
        EDAINLayer(name='EDAIN'),
        backbone_layers,
        target_idx=target_idx,
        target_mean=TARGET_MEAN,
        target_scale=TARGET_SCALE,
        name='EDAIN_LSTM',
    )
    model.compile(optimizer=Adam(learning_rate=BASE_LR), base_lr=BASE_LR, lr_scales=EDAIN_LR_SCALES)
    model(tf.zeros((1, window_size, n_features)), training=False)
    return model

print('LSTM-EDAIN model builder defined.')


## Step 5: Sequences, Metrics, and Window Tuning
Sequences are boundary-safe. Validation window search uses Iverson RMSE on original-scale predictions after Approach A inverse.


In [ ]:
def create_sequences(scaled, raw, window_size, start_idx, end_idx, target_idx):
    """Build sequences; start_idx is the first target index (lookback may cross splits)."""
    x, y = [], []
    for i in range(start_idx, end_idx):
        if i < window_size:
            continue
        x.append(scaled[i - window_size:i, :])
        y.append(raw[i, target_idx])
    x = np.array(x, dtype=np.float32)
    y = np.array(y, dtype=np.float32).reshape(-1, 1)
    return x, y


def make_predictions(model, x_scaled, target_idx, target_mean, target_scale):
    normalized_pred = model.predict(x_scaled, verbose=0)
    raw_avg, raw_std = raw_target_stats_from_scaled_numpy(
        x_scaled, target_idx, target_mean, target_scale
    )
    return inverse_target_raw_numpy(normalized_pred, raw_avg, raw_std)


def _flatten(array):
    return np.asarray(array, dtype=float).reshape(-1)


def iverson_mse(actual, prediction):
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    n = int(mask.sum())
    if n == 0:
        return np.nan
    return float(((prediction - actual) ** 2 * mask).sum() / n)


def iverson_mae(actual, prediction):
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    n = int(mask.sum())
    if n == 0:
        return np.nan
    return float((np.abs(prediction - actual) * mask).sum() / n)


def iverson_rmse(actual, prediction):
    mse = iverson_mse(actual, prediction)
    return float(np.sqrt(mse)) if np.isfinite(mse) else np.nan


def iverson_r2(actual, prediction):
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    if int(mask.sum()) < 2:
        return np.nan
    return float(r2_score(actual[mask], prediction[mask]))


def iverson_rmse_in_original_scale(model, x_scaled, y_true, target_idx, target_mean, target_scale):
    y_pred = make_predictions(model, x_scaled, target_idx, target_mean, target_scale)
    return iverson_rmse(y_true, y_pred)


### LSTM-EDAIN Adam Window Tuning

In [ ]:
window_tuning_results = []
best_window_size = WINDOW_CANDIDATES[0]
best_window_val_rmse = float('inf')

print('Tuning sliding window on validation Iverson RMSE (Approach A inverse)...')
for candidate_window in WINDOW_CANDIDATES:
    x_tune_train, y_tune_train = create_sequences(
        scaled_data, dataset_raw, candidate_window, 0, train_end, target_idx
    )
    x_tune_val, y_tune_val = create_sequences(
        scaled_data, dataset_raw, candidate_window, train_end, val_end, target_idx
    )
    if len(x_tune_val) == 0:
        print(f'  window={candidate_window:3d} -> skipped (0 validation samples)')
        continue

    tune_model = build_model(candidate_window)
    tune_model.fit(
        x_tune_train, y_tune_train,
        validation_data=(x_tune_val, y_tune_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0,
        callbacks=training_callbacks,
    )
    candidate_val_rmse = iverson_rmse_in_original_scale(
        tune_model, x_tune_val, y_tune_val, target_idx, TARGET_MEAN, TARGET_SCALE
    )
    window_tuning_results.append((candidate_window, candidate_val_rmse))
    print(f'  window={candidate_window:3d} -> validation I-RMSE: {candidate_val_rmse:.4f}')
    if candidate_val_rmse < best_window_val_rmse:
        best_window_val_rmse = candidate_val_rmse
        best_window_size = candidate_window
    tf.keras.backend.clear_session()

print(f'\nSelected sliding window: {best_window_size} (validation I-RMSE: {best_window_val_rmse:.4f})')

x_train, y_train = create_sequences(scaled_data, dataset_raw, best_window_size, 0, train_end, target_idx)
x_val, y_val = create_sequences(scaled_data, dataset_raw, best_window_size, train_end, val_end, target_idx)
x_test, y_test = create_sequences(scaled_data, dataset_raw, best_window_size, val_end, len(dataset_raw), target_idx)

assert len(x_val) > 0, f'Validation set empty for window={best_window_size}'
assert len(x_test) > 0, f'Test set empty for window={best_window_size}'
print(f'Train samples: {len(x_train)}, Val samples: {len(x_val)}, Test samples: {len(x_test)}')


In [ ]:
window_df = pd.DataFrame(window_tuning_results, columns=['Window Size', 'Validation I-RMSE'])
plt.figure(figsize=(8, 4))
plt.bar(window_df['Window Size'].astype(str), window_df['Validation I-RMSE'], color='steelblue')
plt.axhline(best_window_val_rmse, color='crimson', linestyle='--', label=f'Best: {best_window_size}')
plt.title('Sliding Window Hyperparameter Tuning — LSTM-EDAIN Adam (Iverson)')
plt.xlabel('Lookback Window (15-min slices)')
plt.ylabel('Validation I-RMSE (num_jobs, actual ≠ 0)')
plt.legend()
plt.tight_layout()
plt.show()


## Step 6: Training LSTM-EDAIN

In [ ]:
model_lstm_edain = build_model(best_window_size)
history_lstm_edain = model_lstm_edain.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=training_callbacks,
    verbose=1,
)
print('LSTM-EDAIN with Adam training completed.')


## Step 7: Predictions and Iverson Metrics

In [ ]:
y_test_actual = y_test
pred_lstm_edain = make_predictions(
    model_lstm_edain, x_test, target_idx, TARGET_MEAN, TARGET_SCALE
)

metrics_values = {
    'LSTM-EDAIN Adam': {
        'I-MSE': iverson_mse(y_test_actual, pred_lstm_edain),
        'I-RMSE': iverson_rmse(y_test_actual, pred_lstm_edain),
        'I-MAE': iverson_mae(y_test_actual, pred_lstm_edain),
        'I-R2': iverson_r2(y_test_actual, pred_lstm_edain),
    }
}

print('Predictions and Iverson metric calculations completed.')
for model_name, m in metrics_values.items():
    print(
        f"  {model_name}: I-MSE = {m['I-MSE']:.4f} | I-RMSE = {m['I-RMSE']:.4f} | "
        f"I-MAE = {m['I-MAE']:.4f} | I-R^2 = {m['I-R2']:.4f}"
    )


In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(y_test_actual, label='Ground Truth', linewidth=1.5)
plt.plot(pred_lstm_edain, label='Prediction', linewidth=1.5, alpha=0.85)
plt.title('LSTM-EDAIN Adam — Ground Truth vs Prediction (Test Set)')
plt.xlabel('Test Sample')
plt.ylabel('num_jobs')
plt.legend()
plt.tight_layout()
plt.show()


## Step 8: Visualizations
* Loss vs Epochs
* Log(Loss) vs Epochs
* Validation Loss vs Epochs
* Final loss bar
* Iverson metrics summary


In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(history_lstm_edain.history['loss'], label='LSTM-EDAIN Adam')
plt.title('Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(np.log10(history_lstm_edain.history['loss']), label='LSTM-EDAIN Adam')
plt.title('Log(Loss) vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Log(Loss)')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(history_lstm_edain.history['val_loss'], label='LSTM-EDAIN Adam')
plt.title('Validation Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(['LSTM-EDAIN Adam'], [history_lstm_edain.history['loss'][-1]], color='steelblue')
plt.title('Final Loss — LSTM-EDAIN Adam')
plt.ylabel('Final Loss')
plt.show()


In [ ]:
metrics_table = [
    [
        model,
        f"{m['I-MSE']:.4f}",
        f"{m['I-RMSE']:.4f}",
        f"{m['I-MAE']:.4f}",
        f"{m['I-R2']:.4f}",
    ]
    for model, m in metrics_values.items()
]
print(tabulate(
    metrics_table,
    headers=['Model', 'I-MSE', 'I-RMSE', 'I-MAE', 'I-R^2'],
    tablefmt='pretty',
))
print('Iverson metrics summary table generated.')
